In [1]:
import re
import csv
from pathlib import Path
from typing import Optional, List, Dict, Tuple
from pymongo import MongoClient
from datetime import timezone
from typing import Optional
import logging
from dotenv import load_dotenv
from bs4 import BeautifulSoup, Tag
from hansard_common import utc_now

In [2]:
LOG_PATH = Path("LOGS/parser.log")
LOG_PATH.parent.mkdir(exist_ok=True)

logging.basicConfig(
    filename=LOG_PATH,
    filemode="a",
    format=">>> %(levelname)s | %(asctime)s | %(message)s",
    level=logging.INFO,
)
logger = logging.getLogger(__name__)

load_dotenv(override=True)

False

In [ ]:
client = MongoClient(
    "mongodb://localhost:27017/",
    tz_aware=True,
    tzinfo=timezone.utc
)

db = client["case-scraping"]

html_col = db["hansard-html-snapshots"]
parsed_col = db["hansard-parsed-rows"]
parsed_col.create_index([("source_url", 1), ("ID", 1)], unique=True)
logger.info("Connecting to MongoDB...")

client.admin.command("ping")

logger.info("MongoDB connection successful")
logger.info(f"Database: {db.name}")
logger.info(f"HTML collection count: {html_col.count_documents({})}")

In [4]:
# normalize whitespace
def clean_text(s: str) -> str:
    s = re.sub(r"\s+", " ", s or "").strip()
    return s

In [5]:
# regex to match speaker lines
SPEAKER_LABEL_RE = re.compile(r"^\s*([^:]{2,200}):") # match start of a line that has characters (2 to 200 for instance) except for :, which ends with a :
"""
From <p class="speakerStart"><strong>Mr. Speaker:</strong> ...</p>
Extract:
    speaker_label: 'Mr. Speaker' (full label before colon)
"""
def parse_speaker(p: Tag) -> Optional[str]:

    full = clean_text(p.get_text(" ", strip=True))
    if not full:
        return None
    
    m = SPEAKER_LABEL_RE.match(full)
    if not m:
        return None
    
    speaker_label = clean_text(m.group(1)) #gets the speaker label before the colon, e.g. "Mr. Speaker (Leader of the Opposition)"
    return speaker_label
    

In [6]:
# generate a unique ID for each row based on date and sequence number
def make_id(d: Optional[str], n: int) -> Optional[str]:
    if not d:
        return None
    ymd = d.replace("-", "")
    return f"{ymd}-{n:07d}"

In [7]:
#in order to be able to differentitate between procedure and continuition of a speaker's speech, i've. created regex that match common procedures. 
PROCEDURAL_START_RE = re.compile(
    r"^(interjections|applause|laughter|prayers|a voice|"
    r"the House adjourned at|the assembly|the committee|the clerk|"
    r"motion (agreed to|carried|negatived)|"
    r"the speaker|the acting speaker|"
    r"it being|pursuant to|ordered that)\b",
    re.I
)
PROCEDURAL_END_RE = re.compile(
    r"pleased to retire\.\s*$",
    re.I
)

def looks_procedural(text: str) -> bool:
    t = clean_text(text)
    if not t:
        return False
    if PROCEDURAL_START_RE.match(t):
        return True
    if PROCEDURAL_END_RE.search(t):
        return True
    return False

In [8]:
# helper for some files where speakerStart tag is not present, but strong tag is used to indicate speaker labels.
def extract_strong_speaker_label(p: Tag) -> Optional[str]:
    strong = p.find("strong")
    if not strong:
        return None

    raw = clean_text(strong.get_text(" ", strip=True))

    # must look like "Speaker Name:"
    if not raw.endswith(":"):
        return None

    raw = raw[:-1].strip()  # remove trailing colon

    if not raw:
        return None

    return raw

In [9]:
# helper function for files that dont have header tags but use all caps paragraphs to indicate headings/subjects.
def is_caps_heading_paragraph(text: str) -> bool:
    t = clean_text(text)
    letters = [ch for ch in t if ch.isalpha()]
    return bool(letters) and ''.join(letters).isupper()

In [ ]:
def parse_transcript_html(html: str) -> List[Dict[str, Optional[str]]]:
    soup = BeautifulSoup(html, "html.parser")  

    # getting the date of the hansard
    date_iso = None
    time_element = soup.find("time", class_="datetime")
    if time_element and time_element.get("datetime"):
        date_iso = time_element["datetime"].split("T")[0].strip()
    if not date_iso:
        # you can parse from canonical URL if needed
        canonical = soup.find("link", rel="canonical")
        if canonical and canonical.get("href"):
            m = re.search(r"/(\d{4}-\d{2}-\d{2})/hansard", canonical["href"])
            date_iso = m.group(1) if m else None
    
    logger.info(f"Starting transcript parse | date={date_iso}")

    transcript = soup.find("div", id="transcript") #gets the transcript container by its id
    if not transcript:
        transcript = soup.find("article")
        if not transcript:
            logger.warning(f"Transcript container missing | date={date_iso}")
            return []
    
    toc = transcript.find("div", id="toc")
    if toc:
        toc.decompose()


    # CSV fields. gets reset with new call to this function for each day's html 
    current_order = None          # OrderofBusiness 
    current_subject = None        # SubjectofBusiness 
    
    # PersonSpeaking
    current_person = None         

    # temporary buffer to accumulate lines of the current speech/intervention
    buffer_lines: List[str] = []

    # list of output rows
    rows: List[Dict[str, Optional[str]]] = []
    seq = 0  # for ID generation

    def flush():
        nonlocal buffer_lines, seq
        if not buffer_lines:
            return
        text = clean_text(" ".join(buffer_lines))
        buffer_lines = []
        if not text:
            return

        seq += 1
        rows.append({
            "ID": make_id(date_iso, seq),
            "Date": date_iso,
            "OrderofBusiness": current_order,
            "SubjectofBusiness": current_subject,
            "PersonSpeaking": current_person or "Procedure",
            "Intervention": text,
        })

    
    for el in transcript.find_all(["h2", "h3", "h4", "p"], recursive=True): #get the headings and paragraphs in the transcript
        if not isinstance(el, Tag):
            continue

        if el.name == "p" and el.find("a", href=True): # in files from the 32nd, 37th, 38th and 39th parliament, there are some paragraphs at the top of the file that contain links to different sectoipns of the hansard. these are not actual interventions and cause parsing issues, so we skip any paragraph that contains links.
            continue
        
        tag = el.name.lower() 

        # Order tracking
        if tag == "h2":
            flush()
            current_order = clean_text(el.get_text(" ", strip=True)) or current_order
            current_subject = None  # reset subject under new order
            continue
        
        # subject tracking
        if tag in {"h3", "h4"}:
            flush()
            current_subject = clean_text(el.get_text(" ", strip=True)) or current_subject
            continue

        # Paragraph handling
        if tag == "p":
            text = clean_text(el.get_text(" ", strip=True))
            if not text:
                continue
            
            #skip lines that only show the time
            if re.fullmatch(r"\d{3,4}", text):
                continue

            if is_caps_heading_paragraph(text):
                flush()
                current_subject = text
                current_person = None
                continue
            
            classes = el.get("class") or []
            speaker_label = None
            if "Procedure" in classes:
                flush()
                saved_person = current_person
                current_person = "Procedure"
                buffer_lines.append(text)
                flush()
                current_person = saved_person
                continue

            if "speakerStart" in classes:
                speaker_label = parse_speaker(el)  
            if not speaker_label:
                speaker_label = extract_strong_speaker_label(el) 

            is_speaker_start = bool(speaker_label) or ("speakerStart" in classes)     

            if is_speaker_start:
                flush()
                # update speaker context if we actually detected one
                if speaker_label:
                    current_person = speaker_label
                    # strip "Label:" from the start of THIS paragraph only
                    text = re.sub(r"^\s*" + re.escape(speaker_label) + r"\s*:\s*", "", text)

                    # if strong tag still caused duplicate label in text, remove it directly from HTML text source
                    strong = el.find("strong")
                    if strong:
                        strong.extract()
                        text = clean_text(el.get_text(" ", strip=True))

                else:
                    logger.warning(f"speakerStart detected but speaker not parsed | date={date_iso} | text={text[:80]}")

                if text:
                    buffer_lines.append(text)

            
            else:
                # Continuation paragraph: same person/procedure continues
                # If we have no speaker context at all, treat as Procedure
                if current_person is None:
                    buffer_lines.append(text)
                else:
                    if looks_procedural(text):
                        flush()
                        # temporarily emit this line as Procedure without killing speaker context
                        saved = current_person
                        current_person = "Procedure"
                        buffer_lines.append(text)
                        flush()
                        # restore speaker context after procedural line
                        current_person = saved
                    else:
                        buffer_lines.append(text)

    flush()
    logger.info(
    f"Finished transcript parse | date={date_iso} | rows={len(rows)}"
)
    return rows
                 

In [17]:
def write_csv(rows: List[Dict[str, Optional[str]]], out_path: str) -> None:
    FIELDNAMES = [
        "ID",
        "Date",
        "OrderofBusiness",
        "SubjectofBusiness",
        "PersonSpeaking",
        "Intervention",
    ]

    Path(out_path).parent.mkdir(parents=True, exist_ok=True)

    with open(out_path, "w", newline="", encoding="utf-8-sig") as f:
        writer = csv.DictWriter(f, fieldnames=FIELDNAMES)
        writer.writeheader()

        for row in rows:
            writer.writerow({k: row.get(k) for k in FIELDNAMES})

In [18]:
def parse_to_mongo():
    logger.info(f"Starting parse run")

    ok = 0
    failed = 0
    total_rows = 0

    cursor = html_col.find({"parsed": {"$ne": True}}, {"url": 1, "content": 1})

    for i, doc in enumerate(cursor, start=1):
        url = doc.get("url")
        content = doc.get("content")

        if content is None:
            failed += 1
            html_col.update_one(
                {"_id": doc["_id"]},
                {"$set": {
                    "parsed": False,
                    "parse_error": "Missing content"
                }}
            )
            logger.warning(f"Missing content | url={url}")
            continue

        if isinstance(content, (bytes, bytearray)):
            html = content.decode("utf-8", errors="ignore")
        elif isinstance(content, str):
            html = content
        else:
            failed += 1
            html_col.update_one(
                {"_id": doc["_id"]},
                {"$set": {
                    "parsed": False,
                    "parse_error": f"Unexpected content type: {type(content)}"
                }}
            )
            logger.warning(f"Unexpected content type | url={url} | type={type(content)}")
            continue

        try:
            rows = parse_transcript_html(html)

            # attach source metadata before inserting
            for row in rows:
                row["source_url"] = url

            if rows:
                parsed_col.insert_many(rows, ordered=False)
            
            html_col.update_one(
                {"_id": doc["_id"]},
                {"$set": {
                    "parsed": True,
                    "parsed_at": utc_now(),
                    "parse_error": None
                }}
            )

            ok += 1
            total_rows += len(rows)
            logger.info(f"Parsed OK | i={i} | url={url} | rows={len(rows)}")

        except Exception as e:
            failed += 1
            html_col.update_one(
                {"_id": doc["_id"]},
                {"$set": {
                    "parsed": False,
                    "parse_error": str(e)
                }}
            )
            logger.exception(f"Parse failed | i={i} | url={url}")

    logger.info(f"Finished parse run | ok={ok} | failed={failed} | total_rows={total_rows}")


In [ ]:
parse_to_mongo()